***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 3 章：干涉测量中的位置天文学](3_0_introduction.ipynb)
    * 上一节：[3.1 赤道坐标、参考历元与星表位置](3_1_equatorial_coordinates.ipynb)
    * 下一节：[3.3 地平坐标、望远镜指向与可观测性](3_3_horizontal_coordinates.ipynb)

***


导入标准模块:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


导入本节所需的专用模块:


In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 3.2 时角、地方恒星时与过中天<a id='pos:sec:ha'></a>

赤经和赤纬给出了天源在惯性参考架中的目录位置，但地面阵列真正关心的问题是：*它此刻在本地天空中的哪一侧？离过中天还有多久？* 这就需要把“固定在星表中的方向”同“随地球自转不断变化的本地子午圈”联系起来。时角和地方恒星时正是完成这一步的核心量。


### 时角的定义

本地子午圈是通过天顶和南北天极的大圆。若一个天源位于子午圈以东，则它尚未过中天；若位于子午圈以西，则它已经过中天。*时角* `H` 就是该天源的时圈与本地子午圈之间沿天赤道量取的角距离。射电天文学中常采用“向西为正”的记号，因此：

\[
H > 0 \quad \Rightarrow \quad \text{源已过中天，位于西侧；}\qquad
H < 0 \quad \Rightarrow \quad \text{源尚未过中天，位于东侧。}
\]

当 `H = 0` 时，天源正在上中天（upper culmination）。在很多阵列上，这通常对应最佳仰角，也常是观测灵敏度和大气稳定性较好的时段。


### 地方恒星时与基本关系

地方恒星时 `LST` 可以理解为“春分点当前的时角”。一旦知道春分点此刻相对于本地子午圈的位置，再结合某个天源的赤经 `\alpha`，就能立刻得到该源的时角：

\[
H = \mathrm{LST} - \alpha.
\]

这里所有量都可以用小时表示，也可以统一换算成角度或弧度。为了避免符号跳变，实际代码中通常会把 `H` 规约到 `(-12\,\mathrm{h}, +12\,\mathrm{h}]` 或 `(-\pi, \pi]` 的区间内。


<a id='pos:fig:hour_angle'></a><img src='figures/hour.svg' width='42%'>

*图 3.2.1*：赤经 `\\alpha`、时角 `H` 与地方恒星时 `LST` 之间的关系。目录坐标 `\\alpha` 是固定的，而 `H` 随地球自转持续变化。


### 为什么恒星时与太阳时不同

日常钟表跟踪的是太阳时；但天文学家观测的多数对象并不是太阳，而是远处恒星和射电源。由于地球在自转的同时还绕太阳公转，一个*太阳日*比一个*恒星日*略长。更准确地说，地方恒星时相对于民用 UTC 的变化速率约为：

\[
\Delta \mathrm{LST} \approx 1.00273790935\, \Delta t_{\mathrm{UTC}}.
\]

这意味着如果只用民用时钟去想象天源在天空中的位置，就会每天多积累大约 4 分钟的偏差。对短时间调度而言，这个差异完全不能忽略。


<a id='pos:fig:sidereal'></a><img src='figures/sidereal.svg' width='52%'>

*图 3.2.2*：由于地球同时自转和公转，恒星日略短于太阳日。正因为如此，地方恒星时会相对于民用时钟逐日提前。


<a id='pos:fig:radec'></a><img src='figures/RADEC.svg' width='90%'>

*图 3.2.3*：黄道与天赤道的相对位置示意。春分点既是黄道与天赤道的交点，也是传统赤道坐标与地方恒星时的零点方向。


### 对干涉观测意味着什么

时角并不是一个抽象的“天球时钟变量”，它直接影响观测几何。

* **过中天时刻**：由 `H = 0` 决定，通常也是源仰角最高的时刻；
* **跟踪区间**：计划观测时常常不是写“从 01:00 到 05:00 UTC”，而是写成“在 `H=-3 h` 到 `H=+2 h` 之间跟踪”；
* **地球自转合成**：同一条基线在不同 `H` 下对应不同投影，因而采样到不同的傅里叶空间位置；
* **质量控制**：当 `|H|` 很大而仰角偏低时，系统温度、增益稳定性和大气相位都可能变差。

换言之，时角把“目录中的静态源位置”变成了“此刻观测几何中的动态位置”。


### 示例：从地方恒星时到过中天时刻

下面用一个理想化例子建立直觉。假设某站在观测开始时的地方恒星时为 `2 h`，目标源赤经为 `5 h 35 m`。我们先忽略所有高阶改正，只保留“恒星时比民用时快一点”这一主效应，看看几个小时内时角如何变化。图中的虚线同时给出了一个理想东西向基线的投影因子 `\cos H`，仅作为地球自转合成几何的预告。


In [ ]:
source_ra_h = 5 + 35 / 60.0
lst0_h = 2.0
sidereal_rate = 1.00273790935
utc_hours = np.linspace(0, 10, 500)

lst_h = (lst0_h + sidereal_rate * utc_hours) % 24.0
hour_angle_h = ((lst_h - source_ra_h + 12.0) % 24.0) - 12.0
projection = np.cos(np.deg2rad(15.0 * hour_angle_h))

transit_index = np.argmin(np.abs(hour_angle_h))
transit_utc_h = utc_hours[transit_index]
print(f'目标源大约在观测开始后 {transit_utc_h:.2f} 小时过中天。')

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(utc_hours, hour_angle_h, color='tab:blue', lw=2, label='时角 H')
ax1.axhline(0.0, color='0.3', ls=':')
ax1.axvline(transit_utc_h, color='tab:blue', ls='--', alpha=0.7)
ax1.set_xlabel('自观测开始起的 UTC 时间 [h]')
ax1.set_ylabel('时角 H [h]', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_ylim(-6, 6)
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(utc_hours, projection, color='tab:orange', lw=2, ls='--', label='示意性投影因子 cos(H)')
ax2.set_ylabel('理想东西向基线的投影因子', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')
ax2.set_ylim(-1.05, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('地方恒星时、时角与过中天时刻的几何直觉')
plt.show()


这个例子说明：一旦地方恒星时已知，时角就能把“当前时刻”直接翻译成“源在本地子午圈的东侧还是西侧，以及离过中天还有多久”。对干涉阵而言，更重要的是，不同的 `H` 对应不同的投影几何，所以一次长时间跟踪并不是重复测同一件事，而是在持续积累新的空间频率采样。


#### 本节要点

* 时角 `H` 把目录中的赤经位置与观测者本地子午圈联系起来，是时间几何中的核心变量；
* 地方恒星时满足 `H = LST - \alpha`，因此一旦已知 `LST` 就能立刻判断源是否过中天；
* 恒星时与太阳时并不相同，这正是短时调度和长期观测计划必须使用 `LST/HA` 的原因；
* 对干涉阵而言，时角不仅决定观测时刻，还决定基线投影如何随地球自转发生变化。


***

下一节：[3.3 地平坐标、望远镜指向与可观测性](3_3_horizontal_coordinates.ipynb)
